# ERA5 Rainfall, Wind and Pressure NetCDF Downloader

This notebook preserves the same AOI workflow used in the previous notebooks:

- Vector AOI: Shapefile, GeoPackage, GeoJSON, or ZIP
- Interactive AOI: draw a rectangle on a map
- Spatial tile-by-tile processing and cache
- Hourly, daily, or monthly output
- Rainfall, 10 m wind, wind speed, wind direction, and surface pressure
- Spatial missing-data detection and nearest-valid-neighbor filling
- Exact all-touched AOI masking that keeps every ERA5 grid cell whose footprint intersects the AOI, including boundary cells
- NetCDF output
- Summary plots

Data source: ERA5-PDS hosted by Microsoft Planetary Computer.

The Planetary Computer ERA5-PDS catalog currently exposes historical data through 2020. The cloud source is Zarr; this notebook subsets it by space/time and writes standard NetCDF `.nc` files locally.

## Install

In [ ]:
%pip install -q xarray "zarr<3" dask adlfs h5netcdf netCDF4 pystac-client planetary-computer geopandas shapely pyproj scipy ipyleaflet ipywidgets matplotlib pandas tqdm
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

## Configuration

In [ ]:
from pathlib import Path
import math, hashlib, json
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer
from shapely.geometry import shape, box
from shapely import intersects, box as shapely_box
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm

MODE = "shapefile"
SHAPEFILE_PATH = ""

START_DATE = "2020-01-01 00:00"
END_DATE = "2020-01-31 23:00"
TIME_MODE = "hourly"

SPATIAL_TILE_DEG = 2.0
ERA5_GRID_DEG = 0.25
OUTPUT_DIR = Path("./ERA5_weather")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_NAME = None
MAX_PREVIEW = 1200

SOURCE_START = pd.Timestamp("1979-01-01 00:00")
SOURCE_END = pd.Timestamp("2020-12-31 23:00")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if TIME_MODE not in {"hourly","daily","monthly"}:
    raise ValueError('TIME_MODE must be "hourly", "daily", or "monthly".')

START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)

if START_TS > END_TS:
    raise ValueError("START_DATE must be earlier than END_DATE.")

if START_TS < SOURCE_START or END_TS > SOURCE_END:
    raise ValueError(f"Planetary Computer ERA5-PDS supports this notebook from {SOURCE_START} through {SOURCE_END}.")

## AOI from Vector File

In [ ]:
def read_vector_aoi(path):
    p = Path(path).expanduser()
    if not p.exists():
        raise FileNotFoundError(p)
    rp = f"zip://{p.resolve()}" if p.suffix.lower() == ".zip" else str(p)
    gdf = gpd.read_file(rp)
    if gdf.empty or gdf.crs is None:
        raise ValueError("The vector file is empty or has no CRS.")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy().to_crs("EPSG:4326")
    geom = gdf.geometry.union_all() if hasattr(gdf.geometry,"union_all") else gdf.geometry.unary_union
    if geom.is_empty:
        raise ValueError("The AOI geometry is empty.")
    return tuple(map(float,geom.bounds)), geom, gdf

if MODE.lower() == "shapefile":
    if not str(SHAPEFILE_PATH).strip():
        SHAPEFILE_PATH = input("Enter Shapefile, GeoPackage, GeoJSON, or ZIP path: ").strip().strip('"').strip("'")
    AOI_BOUNDS, AOI_GEOM, AOI_GDF = read_vector_aoi(SHAPEFILE_PATH)
    print("AOI bounds:", AOI_BOUNDS)

## AOI from Interactive Map

In [ ]:
if MODE.lower() == "map":
    from ipyleaflet import Map, DrawControl, LayersControl
    SELECTED_BOUNDS = None
    SELECTED_GEOM = None
    m = Map(center=(32.0,53.0), zoom=5, scroll_wheel_zoom=True)
    draw = DrawControl()
    draw.rectangle = {"shapeOptions":{"fillOpacity":0.15,"weight":2}}
    draw.polyline = {}
    draw.polygon = {}
    draw.circle = {}
    draw.circlemarker = {}
    draw.marker = {}
    def handle_draw(target, action, geo_json):
        global SELECTED_BOUNDS, SELECTED_GEOM
        if action in ("created","edited"):
            SELECTED_GEOM = shape(geo_json["geometry"])
            SELECTED_BOUNDS = tuple(map(float,SELECTED_GEOM.bounds))
            print("Selected bounds:", SELECTED_BOUNDS)
        elif action == "deleted":
            SELECTED_BOUNDS = SELECTED_GEOM = None
    draw.on_draw(handle_draw)
    m.add_control(draw)
    m.add_control(LayersControl())
    display(m)

## Finalize AOI

In [ ]:
if MODE.lower() == "map":
    if SELECTED_BOUNDS is None:
        raise RuntimeError("Draw a rectangle first.")
    AOI_BOUNDS = SELECTED_BOUNDS
    AOI_GEOM = SELECTED_GEOM if SELECTED_GEOM is not None else box(*AOI_BOUNDS)
    AOI_GDF = None
elif MODE.lower() == "shapefile":
    if "AOI_BOUNDS" not in globals():
        AOI_BOUNDS, AOI_GEOM, AOI_GDF = read_vector_aoi(SHAPEFILE_PATH)
else:
    raise ValueError('MODE must be "shapefile" or "map".')

west,south,east,north = AOI_BOUNDS
if west >= east or south >= north:
    raise ValueError("Invalid AOI bounds.")
if south < -90 or north > 90 or west < -180 or east > 180:
    raise ValueError("AOI must be inside EPSG:4326 bounds.")
# Pad the download bounds by half an ERA5 grid cell.
# This is necessary because a cell can intersect the AOI boundary even when
# its center lies just outside the raw geometry bounds.
EDGE_PAD_DEG = ERA5_GRID_DEG / 2.0
DOWNLOAD_BOUNDS = (
    max(-180.0, west - EDGE_PAD_DEG),
    max(-90.0, south - EDGE_PAD_DEG),
    min(180.0, east + EDGE_PAD_DEG),
    min(90.0, north + EDGE_PAD_DEG),
)

print("Final AOI:", tuple(round(v,6) for v in AOI_BOUNDS))
print("Download bounds (half-cell padded):", tuple(round(v,6) for v in DOWNLOAD_BOUNDS))

## Spatial Tiles

In [ ]:
def spatial_tiles(bounds,size):
    w,s,e,n = map(float,bounds)
    tiles = []
    r = 0
    y = s
    while y < n:
        x = w
        c = 0
        y2 = min(y+size,n)
        while x < e:
            x2 = min(x+size,e)
            tiles.append({"id":f"r{r:03d}_c{c:03d}","bounds":(x,y,x2,y2)})
            x = x2
            c += 1
        y = y2
        r += 1
    return tiles

SPATIAL_TILES = spatial_tiles(DOWNLOAD_BOUNDS,SPATIAL_TILE_DEG)
print("Spatial tiles:",len(SPATIAL_TILES))
for t in SPATIAL_TILES:
    print(t["id"],tuple(round(v,4) for v in t["bounds"]))

## Time Segments

In [ ]:
def month_segments(start,end):
    months = pd.period_range(start=start.to_period("M"),end=end.to_period("M"),freq="M")
    out = []
    for p in months:
        a = max(start,p.start_time)
        b = min(end,p.end_time.floor("h"))
        out.append({"month":str(p),"start":a,"end":b})
    return out

TIME_SEGMENTS = month_segments(START_TS,END_TS)
print("Time segments:",len(TIME_SEGMENTS))
for s in TIME_SEGMENTS:
    print(s["month"],s["start"],"->",s["end"])

## ERA5 Planetary Computer Access

In [ ]:
CATALOG = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

ANALYSIS_VARS = {
    "u10":"eastward_wind_at_10_metres",
    "v10":"northward_wind_at_10_metres",
    "surface_pressure_hpa":"surface_air_pressure"
}

FORECAST_VARS = {
    "rainfall_mm":"precipitation_amount_1hour_Accumulation"
}

def find_item(month,kind):
    items = list(CATALOG.search(
        collections=["era5-pds"],
        datetime=month,
        query={"era5:kind":{"eq":kind}}
    ).items())
    if not items:
        raise RuntimeError(f"No ERA5-PDS {kind} item found for {month}.")
    return items[0]

def open_asset(item,key):
    signed = planetary_computer.sign(item)
    if key not in signed.assets:
        raise KeyError(f"Asset not found: {key}")
    asset = signed.assets[key]
    kwargs = dict(asset.extra_fields.get("xarray:open_kwargs",{}))
    ds = xr.open_dataset(asset.href,**kwargs)
    if key in ds.data_vars:
        da = ds[key]
    else:
        candidates = [v for v in ds.data_vars if not v.endswith("_bounds")]
        if len(candidates) != 1:
            raise RuntimeError(f"Cannot identify variable for asset {key}: {candidates}")
        da = ds[candidates[0]]
    ren = {}
    for d in da.dims:
        dl = d.lower()
        if dl.startswith("time") and d != "time":
            ren[d] = "time"
        elif dl in {"latitude","lat"} and d != "lat":
            ren[d] = "lat"
        elif dl in {"longitude","lon"} and d != "lon":
            ren[d] = "lon"
    da = da.rename(ren)
    if "lon" in da.coords and float(da.lon.max()) > 180:
        da = da.assign_coords(lon=((da.lon+180)%360)-180).sortby("lon")
    return da

## Spatial and Temporal Subset

In [ ]:
def subset_da(da,bounds,start,end):
    w,s,e,n = bounds
    lat_desc = float(da.lat[0]) > float(da.lat[-1])
    lat_slice = slice(n,s) if lat_desc else slice(s,n)
    return da.sel(time=slice(start,end),lat=lat_slice,lon=slice(w,e))

def clean_dataset(ds):
    out = xr.Dataset(
        {
            "rainfall_mm":ds["rainfall_mm"].astype("float32"),
            "u10":ds["u10"].astype("float32"),
            "v10":ds["v10"].astype("float32"),
            "surface_pressure_hpa":ds["surface_pressure_hpa"].astype("float32")
        },
        coords={"time":ds.time,"lat":ds.lat,"lon":ds.lon}
    )
    out["rainfall_mm"].attrs = {"long_name":"Total precipitation","units":"mm"}
    out["u10"].attrs = {"long_name":"10 m eastward wind component","units":"m s-1"}
    out["v10"].attrs = {"long_name":"10 m northward wind component","units":"m s-1"}
    out["surface_pressure_hpa"].attrs = {"long_name":"Surface pressure","units":"hPa"}
    return out

def derive_wind(ds):
    ds["wind_speed"] = np.hypot(ds["u10"],ds["v10"]).astype("float32")
    ds["wind_direction"] = ((270-np.degrees(np.arctan2(ds["v10"],ds["u10"])))%360).astype("float32")
    ds["wind_speed"].attrs = {"long_name":"10 m wind speed","units":"m s-1"}
    ds["wind_direction"].attrs = {"long_name":"Meteorological wind direction","units":"degree"}
    return ds

def aggregate_time(ds,mode):
    if mode == "hourly":
        return derive_wind(ds)
    freq = "1D" if mode == "daily" else "1MS"
    rain = ds["rainfall_mm"].resample(time=freq).sum(min_count=1)
    u = ds["u10"].resample(time=freq).mean()
    v = ds["v10"].resample(time=freq).mean()
    p = ds["surface_pressure_hpa"].resample(time=freq).mean()
    speed = np.hypot(ds["u10"],ds["v10"]).resample(time=freq).mean()
    out = xr.Dataset({"rainfall_mm":rain,"u10":u,"v10":v,"surface_pressure_hpa":p,"wind_speed":speed})
    out["wind_direction"] = ((270-np.degrees(np.arctan2(out["v10"],out["u10"])))%360).astype("float32")
    out["rainfall_mm"].attrs = {"long_name":"Accumulated precipitation","units":"mm"}
    out["u10"].attrs = {"long_name":"Mean 10 m eastward wind component","units":"m s-1"}
    out["v10"].attrs = {"long_name":"Mean 10 m northward wind component","units":"m s-1"}
    out["surface_pressure_hpa"].attrs = {"long_name":"Mean surface pressure","units":"hPa"}
    out["wind_speed"].attrs = {"long_name":"Mean 10 m wind speed","units":"m s-1"}
    out["wind_direction"].attrs = {"long_name":"Meteorological wind direction from mean vector","units":"degree"}
    return out.astype("float32")

## Download and Cache Spatial Tiles

In [ ]:
run_cfg = {
    "bounds":[round(x,6) for x in DOWNLOAD_BOUNDS],
    "aoi_bounds":[round(x,6) for x in AOI_BOUNDS],
    "start":str(START_TS),
    "end":str(END_TS),
    "mode":TIME_MODE,
    "tile_deg":SPATIAL_TILE_DEG,
    "aoi_mask":"all_touched_cell_intersection"
}
RUN_ID = hashlib.sha1(json.dumps(run_cfg,sort_keys=True).encode()).hexdigest()[:12]
RUN_CACHE = CACHE_DIR / RUN_ID
RUN_CACHE.mkdir(parents=True,exist_ok=True)

def valid_nc(path):
    try:
        with xr.open_dataset(path) as ds:
            return all(v in ds for v in ["rainfall_mm","u10","v10","surface_pressure_hpa","wind_speed","wind_direction"])
    except Exception:
        return False

def encoding_for(ds):
    return {v:{"zlib":True,"complevel":4,"dtype":"float32"} for v in ds.data_vars}

TILE_RECORDS = []

for seg in TIME_SEGMENTS:
    an_item = find_item(seg["month"],"an")
    fc_item = find_item(seg["month"],"fc")
    u_src = open_asset(an_item,ANALYSIS_VARS["u10"])
    v_src = open_asset(an_item,ANALYSIS_VARS["v10"])
    p_src = open_asset(an_item,ANALYSIS_VARS["surface_pressure_hpa"])
    r_src = open_asset(fc_item,FORECAST_VARS["rainfall_mm"])

    for tile in tqdm(SPATIAL_TILES,desc=f'ERA5 {seg["month"]}'):
        tag = f'{seg["month"]}_{tile["id"]}_{TIME_MODE}'
        path = RUN_CACHE / f"{tag}.nc"

        if valid_nc(path):
            TILE_RECORDS.append({"month":seg["month"],"tile":tile["id"],"path":path})
            continue

        u = subset_da(u_src,tile["bounds"],seg["start"],seg["end"]).rename("u10")
        v = subset_da(v_src,tile["bounds"],seg["start"],seg["end"]).rename("v10")
        p = (subset_da(p_src,tile["bounds"],seg["start"],seg["end"])/100.0).rename("surface_pressure_hpa")
        r = (subset_da(r_src,tile["bounds"],seg["start"],seg["end"])*1000.0).rename("rainfall_mm")

        r,u,v,p = xr.align(r,u,v,p,join="inner")

        if r.size == 0:
            continue

        ds = clean_dataset(xr.merge([r,u,v,p],compat="override"))
        ds = aggregate_time(ds,TIME_MODE).compute()
        ds.attrs = {
            "source":"ERA5-PDS, Microsoft Planetary Computer",
            "temporal_mode":TIME_MODE,
            "spatial_tile":tile["id"],
            "requested_start":str(START_TS),
            "requested_end":str(END_TS)
        }
        ds.to_netcdf(path,engine="h5netcdf",encoding=encoding_for(ds))
        TILE_RECORDS.append({"month":seg["month"],"tile":tile["id"],"path":path})

    for x in [u_src,v_src,p_src,r_src]:
        try:
            x.close()
        except Exception:
            pass

if not TILE_RECORDS:
    raise RuntimeError("No ERA5 tiles were downloaded.")
print("Cached NetCDF tiles:",len(TILE_RECORDS))
print("Cache:",RUN_CACHE.resolve())

## Merge Spatial Tiles

In [ ]:
def drop_coord_duplicates(ds,name):
    vals = np.asarray(ds[name].values)
    _,idx = np.unique(vals,return_index=True)
    return ds.isel({name:np.sort(idx)})

MONTH_DATASETS = []

for seg in TIME_SEGMENTS:
    paths = [r["path"] for r in TILE_RECORDS if r["month"] == seg["month"]]
    if not paths:
        continue
    opened = [xr.open_dataset(p) for p in paths]
    ds = xr.combine_by_coords(opened,combine_attrs="override")
    ds = drop_coord_duplicates(drop_coord_duplicates(ds,"lat"),"lon")
    ds = ds.sortby("lat").sortby("lon").sortby("time").load()
    for x in opened:
        x.close()
    MONTH_DATASETS.append(ds)

if not MONTH_DATASETS:
    raise RuntimeError("No monthly datasets could be merged.")

weather = xr.concat(MONTH_DATASETS,dim="time").sortby("time")
weather = drop_coord_duplicates(weather,"time")
print(weather)

## Exact AOI Mask (All-Touched Boundary Cells)

In [ ]:
def coordinate_edges(values, default_step=ERA5_GRID_DEG):
    values = np.asarray(values, dtype="float64")
    if values.ndim != 1 or values.size == 0:
        raise ValueError("Coordinate values must be a non-empty 1D array.")
    if values.size == 1:
        half = float(default_step) / 2.0
        return np.array([values[0]-half, values[0]+half], dtype="float64")

    mids = (values[:-1] + values[1:]) / 2.0
    edges = np.empty(values.size + 1, dtype="float64")
    edges[1:-1] = mids
    edges[0] = values[0] - (values[1]-values[0]) / 2.0
    edges[-1] = values[-1] + (values[-1]-values[-2]) / 2.0
    return edges

lon_edges = coordinate_edges(weather.lon.values)
lat_edges = coordinate_edges(weather.lat.values)

# Build the real footprint of every ERA5 grid cell and keep every cell that
# intersects the AOI. This is the raster equivalent of an all_touched mask:
# cells crossed or merely touched by the shapefile boundary are preserved.
CELL_FOOTPRINTS = shapely_box(
    lon_edges[:-1][None,:],
    lat_edges[:-1][:,None],
    lon_edges[1:][None,:],
    lat_edges[1:][:,None]
)
AOI_MASK = np.asarray(intersects(CELL_FOOTPRINTS, AOI_GEOM), dtype=bool)
del CELL_FOOTPRINTS

AOI_MASK_DA = xr.DataArray(AOI_MASK,dims=("lat","lon"),coords={"lat":weather.lat,"lon":weather.lon})
weather = weather.where(AOI_MASK_DA)
print("AOI grid cells (all touched, including boundary):",int(AOI_MASK.sum()))

## Missing Data Fill

In [ ]:
BASE_FILL_VARS = ["rainfall_mm","u10","v10","surface_pressure_hpa","wind_speed"]
REPORT = []

def fill_nearest_2d(a,mask):
    a = np.asarray(a,dtype="float32").copy()
    target = mask & ~np.isfinite(a)
    valid = mask & np.isfinite(a)
    before = int(target.sum())
    if before and valid.any():
        idx = distance_transform_edt(~valid,return_distances=False,return_indices=True)
        a[target] = a[idx[0][target],idx[1][target]]
    a[~mask] = np.nan
    after = int((mask & ~np.isfinite(a)).sum())
    return a,before,after

for var in BASE_FILL_VARS:
    arr = weather[var].values
    total_before = total_after = 0
    for i in range(arr.shape[0]):
        arr[i],b,a = fill_nearest_2d(arr[i],AOI_MASK)
        total_before += b
        total_after += a
    weather[var] = xr.DataArray(arr,dims=("time","lat","lon"),coords={"time":weather.time,"lat":weather.lat,"lon":weather.lon},attrs=weather[var].attrs)
    REPORT.append({"variable":var,"missing_before":total_before,"missing_after":total_after})

weather["wind_direction"] = ((270-np.degrees(np.arctan2(weather["v10"],weather["u10"])))%360).astype("float32")
weather["wind_direction"].attrs = {"long_name":"Meteorological wind direction","units":"degree"}

missing_report = pd.DataFrame(REPORT)
display(missing_report)

## Save Final NetCDF

In [ ]:
if OUTPUT_NAME is None:
    a = START_TS.strftime("%Y%m%d%H")
    b = END_TS.strftime("%Y%m%d%H")
    OUTPUT_NAME = f"ERA5_rain_wind_pressure_{TIME_MODE}_{a}_{b}.nc"

FINAL_PATH = OUTPUT_DIR / OUTPUT_NAME

weather.attrs = {
    "title":"ERA5 Rainfall, Wind and Surface Pressure",
    "source":"ERA5-PDS hosted by Microsoft Planetary Computer",
    "temporal_resolution":TIME_MODE,
    "requested_start":str(START_TS),
    "requested_end":str(END_TS),
    "spatial_resolution":"approximately 0.25 degree",
    "aoi_bounds_wgs84":",".join(map(str,AOI_BOUNDS)),
    "aoi_mask":"all-touched grid-cell footprint intersection; boundary cells retained",
    "missing_fill":"nearest valid spatial neighbor independently for each time step"
}

weather.to_netcdf(FINAL_PATH,engine="h5netcdf",encoding=encoding_for(weather))
missing_report.to_csv(OUTPUT_DIR/"missing_data_report.csv",index=False)

print("NetCDF:",FINAL_PATH.resolve())
print("Size:",f"{FINAL_PATH.stat().st_size/1024**2:.2f} MB")

## NetCDF Summary

In [ ]:
with xr.open_dataset(FINAL_PATH) as ds:
    print(ds)
    print()
    print("Time:",str(ds.time.min().values),"->",str(ds.time.max().values))
    print("Latitude:",float(ds.lat.min()),"->",float(ds.lat.max()))
    print("Longitude:",float(ds.lon.min()),"->",float(ds.lon.max()))

## Rainfall Plot

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

GIF_DIR = OUTPUT_DIR / "gifs"
GIF_DIR.mkdir(exist_ok=True)

def scalar_gif(var, path, title, label, cmap, fps=4):
    with xr.open_dataset(FINAL_PATH) as ds:
        da = ds[var].load()
        lon, lat, times = ds.lon.values, ds.lat.values, pd.to_datetime(ds.time.values)

    vmin = float(np.nanpercentile(da.values, 2))
    vmax = float(np.nanpercentile(da.values, 98))

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(lon, lat, da.isel(time=0), cmap=cmap, shading="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=label)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    def update(i):
        im.set_array(da.isel(time=i).values.ravel())
        ax.set_title(f"{title} | {times[i]}")
        return im,

    ani = FuncAnimation(fig, update, frames=len(times), interval=1000/fps, blit=False)
    ani.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    display(Image(filename=str(path)))

In [ ]:
def wind_gif(path, fps=4):
    with xr.open_dataset(FINAL_PATH) as ds:
        ws = ds.wind_speed.load()
        u = ds.u10.load()
        v = ds.v10.load()
        lon, lat, times = ds.lon.values, ds.lat.values, pd.to_datetime(ds.time.values)

    vmin = float(np.nanpercentile(ws.values, 2))
    vmax = float(np.nanpercentile(ws.values, 98))
    X, Y = np.meshgrid(lon, lat)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.pcolormesh(lon, lat, ws.isel(time=0), cmap="viridis", shading="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label="Wind Speed (m/s)")

    step = max(1, int(max(len(lat), len(lon)) / 20))
    q = ax.quiver(
        X[::step, ::step], Y[::step, ::step],
        u.isel(time=0).values[::step, ::step],
        v.isel(time=0).values[::step, ::step]
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    def update(i):
        im.set_array(ws.isel(time=i).values.ravel())
        q.set_UVC(
            u.isel(time=i).values[::step, ::step],
            v.isel(time=i).values[::step, ::step]
        )
        ax.set_title(f"10 m Wind | {times[i]}")
        return im, q

    ani = FuncAnimation(fig, update, frames=len(times), interval=1000/fps, blit=False)
    ani.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    display(Image(filename=str(path)))

In [ ]:
RAINFALL_GIF = GIF_DIR / f"rainfall_{TIME_MODE}.gif"

scalar_gif(
    "rainfall_mm",
    RAINFALL_GIF,
    "Rainfall",
    "Rainfall (mm)",
    "Blues",
    fps=4
)

## Wind Plot

In [ ]:
WIND_GIF = GIF_DIR / f"wind_{TIME_MODE}.gif"

wind_gif(
    WIND_GIF,
    fps=4
)

## Pressure Plot

In [ ]:
PRESSURE_GIF = GIF_DIR / f"pressure_{TIME_MODE}.gif"

scalar_gif(
    "surface_pressure_hpa",
    PRESSURE_GIF,
    "Surface Pressure",
    "Pressure (hPa)",
    "Spectral_r",
    fps=4
)

## Time Series

In [ ]:
with xr.open_dataset(FINAL_PATH) as ds:
    spatial_dims = ("lat","lon")
    rain_ts = ds["rainfall_mm"].mean(spatial_dims,skipna=True)
    wind_ts = ds["wind_speed"].mean(spatial_dims,skipna=True)
    pressure_ts = ds["surface_pressure_hpa"].mean(spatial_dims,skipna=True)

    plt.figure(figsize=(12,4))
    rain_ts.plot()
    plt.title("AOI Mean Rainfall")
    plt.ylabel("Rainfall (mm)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12,4))
    wind_ts.plot()
    plt.title("AOI Mean Wind Speed")
    plt.ylabel("Wind speed (m/s)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12,4))
    pressure_ts.plot()
    plt.title("AOI Mean Surface Pressure")
    plt.ylabel("Pressure (hPa)")
    plt.tight_layout()
    plt.show()

## Output Inventory

In [ ]:
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(p)